# Module 8 – Artificial Neural Networks (ANN)
## Instructor Solution Notebook

This notebook mirrors the **student practice notebook** but includes:
- A reference implementation of all steps.
- Example hyperparameter choices and typical outputs.
- Instructor-oriented comments and talking points.

**Main components:**
1. Data loading and preprocessing (Breast Cancer dataset).
2. Baseline MLP model.
3. MLP architecture comparison (different depths).
4. RBF network using K-Means + Ridge.
5. Summary comparison and reflection prompts.


In [ ]:
# ==============================
# 1. Imports & Global Settings
# ==============================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RANDOM_STATE = 42
np.set_printoptions(precision=3, suppress=True)

print("Libraries imported successfully.")

In [ ]:
# =======================================
# 2. Load & Inspect the Breast Cancer Data
# =======================================

data = load_breast_cancer()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

print("Dataset shape (samples, features):", X.shape)
print("Target classes:", target_names)
print("Class distribution (counts):", np.bincount(y))
print("First 5 feature names:", feature_names[:5])

# Instructor note:
# Class 0 = malignant, Class 1 = benign (in this dataset).
# Often accuracy is high for both simple and complex models, which is a good
# opportunity to discuss calibration, recall for malignant cases, etc.

In [ ]:
# =======================================
# 3. Train / Test Split + Feature Scaling
# =======================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training set shape:", X_train_scaled.shape)
print("Test set shape     :", X_test_scaled.shape)
print("Mean of first scaled feature (train): {:.3f}".format(X_train_scaled[:, 0].mean()))
print("Std of first scaled feature  (train): {:.3f}".format(X_train_scaled[:, 0].std()))

# Instructor note:
# These means/stds should be close to 0 and 1 respectively, confirming scaling.

In [ ]:
# =======================================
# 4. Baseline MLP Classifier
# =======================================

hidden_layers = (32, 16)
activation = "relu"

mlp = MLPClassifier(
    hidden_layer_sizes=hidden_layers,
    activation=activation,
    solver="adam",
    max_iter=300,
    random_state=RANDOM_STATE,
)

print("Training baseline MLP with layers =", hidden_layers, "and activation =", activation)
mlp.fit(X_train_scaled, y_train)

y_pred_mlp = mlp.predict(X_test_scaled)

print("\n=== Baseline MLP Results ===")
acc_mlp = accuracy_score(y_test, y_pred_mlp)
print("Test Accuracy: {:.3f}".format(acc_mlp))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_mlp, target_names=target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_mlp))

# Instructor talking points:
# - Accuracy is typically around 0.95+ on this dataset with a reasonable MLP.
# - Highlight precision/recall for the malignant class (index 0).
# - Discuss whether misclassifying malignant as benign is more serious.

In [ ]:
# =======================================================
# 5. MLP Architecture Comparison (Depth / Width)
# =======================================================

architectures = [
    (16,),                # shallow
    (32, 16),             # baseline
    (64, 32, 16),         # deeper
    (64, 64, 32, 16),     # even deeper
    (32, 32, 32, 16, 16), # more layers, moderate width
]

results_mlp = []

for arch in architectures:
    model = MLPClassifier(
        hidden_layer_sizes=arch,
        activation="relu",
        solver="adam",
        max_iter=400,
        random_state=RANDOM_STATE,
    )
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results_mlp.append((arch, acc))

print("=== MLP Architecture Comparison (ReLU, Adam) ===")
for arch, acc in results_mlp:
    print(f"Architecture {arch} -> Test Accuracy = {acc:.3f}")

# Instructor talking points:
# - Emphasize that on tabular datasets with limited size, deeper networks
#   do not always improve performance.
# - Use this section to discuss bias–variance trade-off and overfitting.
# - If any architecture underperforms, discuss potential convergence issues.

In [ ]:
# =======================================
# 6. Optional: MLP Loss Curve Visualization
# =======================================

if hasattr(mlp, "loss_curve_"):
    plt.figure()
    plt.plot(mlp.loss_curve_)
    plt.title("Baseline MLP Training Loss Curve")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid(True)
    plt.show()

# Instructor note:
# - If the loss curve plateaus early, consider increasing max_iter.
# - If the curve is noisy, discuss learning rate and optimizer choice.

In [ ]:
# =======================================
# 7. RBF Network – K-Means + Ridge
# =======================================

def rbf_kernel(X, centers, gamma):
    """Compute Gaussian RBF activations for all samples and centers."""
    dists = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)
    return np.exp(-gamma * dists**2)

# Baseline number of centers
n_centers = 25

print("Fitting K-Means with n_centers =", n_centers)
kmeans = KMeans(n_clusters=n_centers, random_state=RANDOM_STATE)
kmeans.fit(X_train_scaled)
centers = kmeans.cluster_centers_

gamma = 1.0 / (2 * np.var(X_train_scaled))
print("Gamma used for RBF =", gamma)

Phi_train = rbf_kernel(X_train_scaled, centers, gamma)
Phi_test = rbf_kernel(X_test_scaled, centers, gamma)

print("RBF feature matrix shape (train):", Phi_train.shape)

rbf_model = Ridge(alpha=1.0)
rbf_model.fit(Phi_train, y_train)

y_scores_rbf = rbf_model.predict(Phi_test)
y_pred_rbf = (y_scores_rbf >= 0.5).astype(int)

print("\n=== RBF Network Results (n_centers = {}) ===".format(n_centers))
acc_rbf = accuracy_score(y_test, y_pred_rbf)
print("Test Accuracy: {:.3f}".format(acc_rbf))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rbf, target_names=target_names))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rbf))

# Instructor talking points:
# - Clarify the two-stage nature: unsupervised center selection + supervised linear readout.
# - Compare capacity vs. interpretability vs. training speed with MLP.

In [ ]:
# =======================================
# 8. RBF Sensitivity to Number of Centers
# =======================================

def evaluate_rbf_for_centers(centers_list):
    results = []
    for n_c in centers_list:
        km = KMeans(n_clusters=n_c, random_state=RANDOM_STATE)
        km.fit(X_train_scaled)
        c = km.cluster_centers_
        Phi_tr = rbf_kernel(X_train_scaled, c, gamma)
        Phi_te = rbf_kernel(X_test_scaled, c, gamma)
        model = Ridge(alpha=1.0)
        model.fit(Phi_tr, y_train)
        y_sc = model.predict(Phi_te)
        y_pr = (y_sc >= 0.5).astype(int)
        acc = accuracy_score(y_test, y_pr)
        results.append((n_c, acc))
    return results

centers_to_try = [5, 10, 25, 40, 60]
rbf_results = evaluate_rbf_for_centers(centers_to_try)

print("=== RBF Accuracy vs Number of Centers ===")
for n_c, acc in rbf_results:
    print(f"n_centers = {n_c:2d} -> Test Accuracy = {acc:.3f}")

# Instructor note:
# - Very small n_centers -> underfitting (RBF basis is too coarse).
# - Very large n_centers -> potential overfitting and higher computation.
# - Use this to contrast how 'capacity' is controlled in RBF vs MLP.

In [ ]:
# =======================================
# 9. Final Summary & Discussion Prompts
# =======================================

print("Baseline MLP accuracy : {:.3f}".format(acc_mlp))
print("Baseline RBF accuracy : {:.3f}".format(acc_rbf))

print("\nSuggested discussion questions:")
print("1) Which model is more flexible on this dataset, and why?")
print("2) How does model depth (for MLP) compare to number of centers (for RBF)")
print("   as a way of controlling capacity?")
print("3) In real medical applications, what additional metrics or constraints")
print("   should we consider beyond accuracy (e.g., sensitivity/recall for the")
print("   malignant class, interpretability, robustness)?")
print("4) How would you explain the difference between global activations (MLP)")
print("   and local activations (RBF) in simple, non-technical language to a")
print("   healthcare stakeholder?")